In [669]:
# Brian Loch (4/26/2026)
# Packages
import pandas as pd
import random
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import RandomForestClassifier as RFC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

In [671]:
# --- Global Configuration ---
indep_vars = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 'height_m', 'weight_kg']
Seed = 5596

# --- Load and Prepare Data ---
df = pd.read_csv('../data/pokemon_complete.csv')
X = df[indep_vars]

# Prepare Multi-Label Target (Types 1 & 2)
df['all_types'] = df[['type_1', 'type_2']].apply(
    lambda x: [t for t in x if pd.notna(t)], axis=1
)
mlb = MultiLabelBinarizer()
Y_multi = mlb.fit_transform(df['all_types'])

# Prepare Single-Label Target (Type 1 only)
le = LabelEncoder()
Y_single = le.fit_transform(df['type_1'])

# --- Accuracy Check ---
# Best accuracy so far have been with,
# 66.67% for single type guesses,
# Seed = 5596, test_size = 0.0111 and n_estimators= 114, Y_single
# 25.00% for dual type guesses,
# Seed = 5596, test_size = 0.0112 and n_estimators= 1, Y_multi
# Y_Multi and Y_Single are interchangable as a variable in the line below
# If you are swapping between either variable, just run this box and then continue running tests
X_train_m, X_test_m, Y_train_m, Y_test_m = train_test_split(
    X, Y_multi, test_size=0.0112, random_state=Seed, shuffle=True
)

model_multi = RFC(n_estimators=1, random_state=Seed)
model_multi.fit(X_train_m, Y_train_m)

accuracy = accuracy_score(Y_test_m, model_multi.predict(X_test_m))
print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 25.00%


In [657]:
# --- Prediction Functions ---

def test_pokemon_by_row(pkmn_row, threshold=0.20):
    sample = pd.DataFrame([pkmn_row[indep_vars].values], columns=indep_vars)
    raw_probs = model_multi.predict_proba(sample)
    
    # --- Format Detection ---
    # If raw_probs is a list, it's Multi-Label (y_multi)
    if isinstance(raw_probs, list):
        probs = [p[0][1] for p in raw_probs]
        current_classes = mlb.classes_
    # If it's a single numpy array, it's Single-Label (y_single)
    else:
        probs = raw_probs[0]
        current_classes = le.classes_
    
    # --- Shared Logic ---
    type_probabilities = list(zip(current_classes, probs))
    type_probabilities.sort(key=lambda x: x[1], reverse=True)
    
    # Limit to top 2 results that meet the threshold
    predicted_types = [
        name for name, prob in type_probabilities[:2] 
        if prob >= threshold
    ]
    
    actual_types = pkmn_row['all_types']

    print("-" * 40)
    print(f"POKEMON: {pkmn_row['name']} (No. {pkmn_row['pokedex_number']})")
    print("-" * 40)
    print(f"ACTUAL TYPES   : {actual_types}")
    print(f"PREDICTED TYPES: {predicted_types}")
    print("-" * 40)

def test_random_pokemon(threshold=0.20):
    random_idx = random.randint(0, len(df) - 1)
    test_pokemon_by_row(df.iloc[random_idx], threshold)

In [668]:
# --- Run Test ---
test_random_pokemon()

----------------------------------------
POKEMON: Dartrix (No. 723)
----------------------------------------
ACTUAL TYPES   : ['Grass', 'Flying']
PREDICTED TYPES: ['Flying', 'Grass']
----------------------------------------
